# Td-Tp 7 : Optimisation des hyperparamètes

Cette fois, ces exercices prennent comme exemple le réseau neuronal entièrement connecté dans Td-Tp 3 pour optimiser les hyperparamètres et éviter le surapprentisage pendant le processus de formation du réseau.

Ici, nous mettons le code que tout le monde a complété dans le quatrième Tp directement devant pour votre référence ultérieure.

In [ ]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import h5py
import scipy
from PIL import Image
from scipy import ndimage

%matplotlib inline
%load_ext autoreload
%autoreload 2

def load_dataset():
    train_dataset = h5py.File('datasets/train_catvnoncat.h5', "r")
    train_set_x_orig = np.array(train_dataset["train_set_x"][:]) # your train set features
    train_set_y_orig = np.array(train_dataset["train_set_y"][:]) # your train set labels

    test_dataset = h5py.File('datasets/test_catvnoncat.h5', "r")
    test_set_x_orig = np.array(test_dataset["test_set_x"][:]) # your test set features
    test_set_y_orig = np.array(test_dataset["test_set_y"][:]) # your test set labels

    classes = np.array(test_dataset["list_classes"][:]) # the list of classes
    
    train_set_y_orig = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
    test_set_y_orig = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))
    
    return train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig, classes


train_set_x_orig, train_set_y, test_set_x_orig, test_set_y, classes = load_dataset()


n_train = train_set_x_orig.shape[0]
n_test = test_set_x_orig.shape[0]
num_px = train_set_x_orig.shape[1]  


train_set_x_flatten = train_set_x_orig.reshape(train_set_x_orig.shape[0], -1).T
test_set_x_flatten = test_set_x_orig.reshape(test_set_x_orig.shape[0], -1).T 


train_set_x = train_set_x_flatten / 255.
test_set_x = test_set_x_flatten / 255.

In [ ]:
def sigmoid(z):  
    return 1/(1+np.exp(-z))


def initialize_with_zeros(dim):
    w = np.zeros([dim,1])
    b = 0.0
    return w, b


def propagate(w, b, X, Y):
    n = X.shape[1]
    
    ## PROPAGATION AVANT (DE X AU COÛT)
    hat_Y = sigmoid(np.dot(w.T,X)+b)
    cost = -1/n * (np.dot(Y,np.log(hat_Y).T) + np.dot((1-Y),np.log(1 - hat_Y).T))

    ## PROPAGATION ARRIÈRE (POUR TROUVER LE GRAD)
    dw = 1/n * np.dot(X,(hat_Y-Y).T)
    db = 1/n * np.sum(hat_Y-Y)
    
    cost = np.squeeze(np.array(cost))
    
    grads = {"dw": dw,
             "db": db}
    return grads, cost


def optimize(w, b, X, Y, num_iterations=100, learning_rate=0.009, print_cost=False):
    w = copy.deepcopy(w)
    b = copy.deepcopy(b)
    
    costs = []
    
    for i in range(num_iterations):
        ## Calcul du coût et du gradient
        grads, cost = propagate(w, b, X, Y)
        
        ## Récupérer les dérivés de grads
        dw = grads["dw"]
        db = grads["db"]
        
        ## mise à jour
        w += -learning_rate * dw
        b += -learning_rate * db
        
        ## Enregistrer les coûts
        if i % 100 == 0:
            costs.append(cost)
        
            ## Imprimer le coût toutes les 100 itérations d'entraînement
            if print_cost:
                print ("Coût après itération %i: %f" %(i, cost))
    
    params = {"w": w,
              "b": b}
    
    grads = {"dw": dw,
             "db": db}    
    return params, grads, costs


def predict(w, b, X):
    n = X.shape[1]
    Y_prediction = np.zeros((1, n))
    w = w.reshape(X.shape[0], 1)
    
    ## Calculer le vecteur "hat_Y" prédisant les probabilités qu'un chat soit présent dans l'image
    hat_Y = sigmoid(np.dot(w.T,X)+b)
    
    for i in range(hat_Y.shape[1]):
        ## Convertir les probabilités hat_Y[0,i] en prédictions réelles p[0,i]
        if hat_Y[0, i] > 0.5 :
            Y_prediction[0,i] = 1
        else:
            Y_prediction[0,i] = 0
    return Y_prediction


def model(X_train, Y_train, X_test, Y_test, num_iterations=2000, learning_rate=0.5, print_cost=False):
    w, b = initialize_with_zeros(X_train.shape[0])
    
    ## Descente de gradient
    params, grads, costs = optimize(w, b, X_train, Y_train, num_iterations, learning_rate, print_cost)
    
    ## Récupérer les paramètres w et b du dictionnaire `params`
    w = params["w"]
    b = params["b"]
        
    ## Prédire les exemples d'ensembles de test/d'entraînement
    Y_prediction_test = predict(w, b, X_test)
    Y_prediction_train = predict(w, b, X_train)

    ## Imprimer les erreurs d'entraînement/de test
    pe = 100 - np.mean(np.abs(Y_prediction_train - Y_train)) * 100
    pt = 100 - np.mean(np.abs(Y_prediction_test - Y_test)) * 100
    if print_cost:
        print("précision des entraînements : {} %".format(pe))
        print("précision des tests : {} %".format(pt))

    
    d = {"costs": costs,
         "Y_prediction_test": Y_prediction_test, 
         "Y_prediction_train" : Y_prediction_train, 
         "w" : w, 
         "b" : b,
         "learning_rate" : learning_rate,
         "num_iterations": num_iterations,
         "res" : np.sign(pe-pt)*pt}
    return d

## Matrice de confusion
Une fois que vous avez construit un modèle de classification, vous devez évaluer la qualité des prédictions faites par ce modèle. Alors, comment définissez-vous les « bonnes » prédictions ?

Certaines mesures de performance nous aident à améliorer nos modèles. Explorons les différences entre elles pour un problème de classification binaire :

Considérez la matrice de confusion suivante pour un problème de classification :

<img src="confusion.png" style="width:500px">

Voici maintenant les mesures fondamentales pour les données ci-dessus :
1. **Précision (Precision)** : Il s'agit de la mesure des cas positifs correctement identifiés parmi tous les cas positifs prédits. Elle est utile lorsque les coûts des faux positifs sont élevés.
$$precision= \frac{True\, Positives\, (TP)}{True\, Positives\, (TP) + False\, Positives\, (FP)} = \frac{20}{20+5} = 80\%$$

2. **Rappel (Recall)** : C'est la mesure des cas positifs correctement identifiés parmi l'ensemble des cas positifs réels. Elle est importante lorsque le coût des faux négatifs est élevé.
$$recall= \frac{True\, Positives\, (TP)}{True\, Positives\, (TP)+False\, Negatives\, (FN)} = \frac{20}{20+10} \approx 66.6\% $$

3. **Exactitude (Accuracy)** : L'une des mesures les plus évidentes, c'est la mesure de tous les cas correctement identifiés. Elle est surtout utilisée lorsque toutes les classes sont d’égale importance.ous les cas correctement identifiés. Il est surtout utilisé lorsque toutes les classes sont d’égale importance.
$$accuracy= \frac{True\, Positives\, (TP)+True\, Negatives\, (TN)}{True\, Positives\, (TP)+False\, Positives\, (FP)+True\, Negatives\, (TN)+False\, Negatives\, (FN)} = \frac{20+65}{20+5+65+10} = 85\% $$

## Fonction d'objectif d'optimisation personnalisée (Par exemple, précision, score F1, etc.)**

1. [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)

`accuracy_score` est une fonction de la bibliothèque Scikit-learn qui calcule la précision d'un modèle de classification. Si $\hat{y}_i$ est la valeur prédite du i-ème échantillon et $y_i$ est la vraie valeur correspondante, alors la fraction de prédictions correctes sur $n_{samples}$ est définie comme $$accuracy(y ,\hat{y}) = \frac{1}{n_{samples}}\sum_{i=0}^{n_{samples}-1}\mathbb{1}\left(\hat{y}_i= y_i\right)$$ où $\mathbb{1}(x)$ est la fonction d'indicateur. 

```python
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_true, y_pred)
```

2. [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
La précision (precision) fait référence à la proportion d’échantillons prédits comme positifs par le modèle qui le sont réellement : $$precision= \frac{True\, Positives\, (TP)}{True\, Positives\, (TP) + False\, Positives\, (FP)}$$
```python
from sklearn.metrics import precision_score
precision = precision_score(y_true, y_pred)
```


3. [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
Le rappel (recall) fait référence à la proportion d'échantillons réellement positifs qui sont correctement prédits comme positifs par le modèle : $$recall= \frac{True\, Positives\, (TP)}{True\, Positives\, (TP)+False\, Negatives\, (FN)}$$
```python
from sklearn.metrics import recall_score
recall = recall_score(y_true, y_pred)
```

4. [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score)

`f1_score` est un indicateur utilisé pour évaluer les performances des modèles de classification, particulièrement adapté aux ensembles de données déséquilibrés. Sa formule de calcul est : $$F1-score = 2 \times \frac{precision + recall}{precision \times recall}$$ Ici, 
```python
from sklearn.metrics import f1_score
f1 = f1_score(y_true, y_pred)
```

5. [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
```python
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(y_true, y_score)
```

# Table des matières
- [I - Réalisation sans l'aide de Scikit Learn](#1)
    - [1.1 - Optimisation](#1.1)
        - [1.1.1 - Recherche par grille](#1.1.1)
        - [1.1.2 - Recherche aléatoire](#1.1.2)
            - [Exercice 1 : Recherche aléatoire](#exo-1)
    - [1.2 - Surapprentisage](#1.2)
        - [1.2.1 - LASSO](#1.2.1)
        - [1.2.2 - Ridge](#1.2.2)
            - [Exercice 2 : Ridge](#exo-2)
        - [1.2.3 - Recherche par grille avec validation croisée](#1.2.3)
        - [1.2.4 - Recherche aléatoire avec validation croisée](#1.2.4)
            - [Exercice 3 : Recherche aléatoireavec validation croisée](#exo-3)
- [II - Réalisation à l'aide de Scikit Learn](#2)
    - [2.1 - Optimisation](#2.1)
        - [2.1.1 - Recherche par grille](#2.1.1)
        - [2.1.2 - Recherche aléatoire](#2.1.2)
        - [2.1.3 - Optimisation bayésienne](#2.1.3)
    - [2.2 - Surapprentisage](#2.2)
        - [2.2.1 - LASSO et Ridge](#2.2.1)
- [III - Réseaux Adverses Génératifs (GAN)](#3)
    - [3.1 - Générateur](#3.1)
        - [Exercice 4 : Generator](#exo-4)
    - [3.2 - Discriminateur](#3.2)
        - [Exercice 5 : Discriminator](#exo-5)
    - [3.3 - Boucle d'entraînement du GAN](#3.3)
        - [Exercice 6 : train_gan](#exo-6) 

<a name='1'></a>
# I - Réalisation sans l'aide de Scikit Learn
<a name='1.1'></a>
## 1 - Optimisation
<a name='1.1.1'></a>
### 1.1 - Recherche par grille

Les combinaisons qui doivent être évaluées par l'utilisateur sont testées avec `GridSearchCV` dans la bibliothèque Sklearn. En fait, le modèle s'adapte à chaque combinaison individuellement, révélant les meilleurs résultats et paramètres. Par exemple, lorsque nous considérons LogisticRegression, si 4 valeurs différentes sont sélectionnées pour taux d'apprentisage et 4 valeurs différentes sont sélectionnées pour le nombre total d'itération, le modèle s'ajustera 16 fois et le résultat de chacune sera représenté. Créons maintenant une recherche par grille sur l'ensemble de données sans utiliser la bibliothèque sklearn :

In [ ]:
## Recherche par grille

best_lr = 0
for lr in [0.001, 0.005, 0.01, 0.05]:
    for ni in [500, 1000, 2000, 3000]:
        logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=ni, learning_rate=lr, print_cost=False)
        lr_score = logistic_regression_model["res"]
        print("num_iterations: ", ni, "learning_rate: ", lr,'res: {:.3f}'.format(lr_score))
        if lr_score > best_lr:
            best_lr = lr_score
            best_lr_combination = (ni, lr)
print("best score LogisticRegression",best_lr)
print("best num_iterations and learning_rate",best_lr_combination)

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=500, learning_rate=0.01, print_cost=True)

<a name='1.1.2'></a>
### 1.2 - Recherche aléatoire

Par rapport à l'algorithme de recherche par grille précédent, dans l'algorithme de recherche aléatoire, nous prenons le même nombre de groupes de paramètres, mais les groupes de paramètres ici sont des nombres aléatoires dans un intervalle donné. Veuillez utiliser `np.random.seed(1)` et `np.random.randn` pour compléter cette partie du code.

<img src="grille_aleatoire.png" style="width:500px">

<a name='exo-1'></a>
#### Exercice 1 : Recherche aléatoire
**Complétez cette partie du code en vous basant sur le code ci-dessus.**

In [ ]:
## Recheche aléatoire
# 16 points verifiés
# 0.001 <= lr < 0.05
# 500 <= ni < 3000

np.random.seed(1)





**Output attendu** :
<table>
    <tr>
        <td>
            best score LogisticRegression
        </td>
        <td>
            74.0
        </td>
    </tr>
    <tr>
        <td>best num_iterations and learning_rate
        </td>
        <td>
            (931, 0.002913684378411236)
        </td>
    </tr>
</table>

<a name='1.2'></a>
## 2 - Surapprentisage

<a name='1.2.1'></a>
### 2.2 - LASSO
- L'implémentation de LASSO consiste à ajouter un terme de régularisation L1 à la fonction de coût. Ce terme de régularisation est le produit de la somme des valeurs absolues du vecteur de poids $w$ et d'un paramètre d'ajustement $\lambda$. Plus $\lambda$ est grand, plus l'effet de la régularisation est fort, ce qui peut rendre le modèle plus parcimonieux, c'est-à-dire que les coefficients des caractéristiques tendent vers zéro.
$$J = L+s = L + \frac{\lambda}{n}\sum_{i}|w_i|$$

- Une caractéristique distinctive de LASSO est sa capacité à effectuer une sélection de caractéristiques, c'est-à-dire qu'il peut réduire à zéro les coefficients des caractéristiques moins importantes, simplifiant ainsi le modèle et améliorant son pouvoir explicatif.

In [ ]:
import numpy as np
import copy

def sigmoid(z):  
    return 1/(1+np.exp(-z))

def initialize_with_zeros(dim):
    w = np.zeros([dim,1])
    b = 0.0
    return w, b

def propagate(w, b, X, Y, lambd):
    n = X.shape[1]
    
    ## Forward propagation (from X to cost)
    hat_Y = sigmoid(np.dot(w.T,X)+b)
    cost = -1/n * (np.dot(Y,np.log(hat_Y).T) + np.dot((1-Y),np.log(1 - hat_Y).T)) + lambd * np.sum(np.abs(w))
    
    ## Backward propagation (to find gradient)
    dw = 1/n * np.dot(X,(hat_Y-Y).T) + (lambd / n) * np.sign(w)
    db = 1/n * np.sum(hat_Y-Y)
    
    cost = np.squeeze(np.array(cost))
    
    grads = {"dw": dw,
             "db": db}
    return grads, cost

def optimize(w, b, X, Y, num_iterations=100, learning_rate=0.009, lambd=0.1, print_cost=False):
    w = copy.deepcopy(w)
    b = copy.deepcopy(b)
    
    costs = []
    
    for i in range(num_iterations):
        ## Calculate cost and gradient
        grads, cost = propagate(w, b, X, Y, lambd)
        
        ## Retrieve derivatives from grads
        dw = grads["dw"]
        db = grads["db"]
        
        ## Update
        w += -learning_rate * dw
        b += -learning_rate * db
        
        ## Save costs
        if i % 100 == 0:
            costs.append(cost)
        
            ## Print cost every 100 training iterations
            if print_cost:
                print ("Cost after iteration %i: %f" %(i, cost))
    
    params = {"w": w,
              "b": b}
    
    grads = {"dw": dw,
             "db": db}    
    return params, grads, costs

def model(X_train, Y_train, X_test, Y_test, num_iterations=2000, learning_rate=0.5, lambd=0.1, print_cost=False):
    w, b = initialize_with_zeros(X_train.shape[0])
    
    ## Gradient descent
    params, grads, costs = optimize(w, b, X_train, Y_train, num_iterations, learning_rate, lambd, print_cost)
    
    ## Retrieve w and b parameters from dictionary `params`
    w = params["w"]
    b = params["b"]
        
    ## Predict examples from test/train sets
    Y_prediction_test = predict(w, b, X_test)
    Y_prediction_train = predict(w, b, X_train)

    ## Print training/test errors
    train_accuracy = 100 - np.mean(np.abs(Y_prediction_train - Y_train)) * 100
    test_accuracy = 100 - np.mean(np.abs(Y_prediction_test - Y_test)) * 100
    if print_cost:
        print("Training accuracy: {} %".format(train_accuracy))
        print("Test accuracy: {} %".format(test_accuracy))
    
    d = {"costs": costs,
         "Y_prediction_test": Y_prediction_test, 
         "Y_prediction_train" : Y_prediction_train, 
         "w" : w, 
         "b" : b,
         "learning_rate" : learning_rate,
         "num_iterations": num_iterations,
         "train_accuracy" : train_accuracy,
         "test_accuracy" : test_accuracy,
         "res" : np.sign(train_accuracy-test_accuracy)*test_accuracy}
    return d

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, 
                                  num_iterations=1000, 
                                  learning_rate=0.01, 
                                  lambd = 0.2,
                                  print_cost=True)

In [ ]:
best_score = 0
for l in [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]:
    logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y,
                                      num_iterations=1000,
                                      learning_rate=0.01, 
                                      lambd = l,
                                      print_cost=False)
    lr_score = logistic_regression_model["res"]
    print("lambda: ", l, 'res: {:.3f}'.format(lr_score))
    if lr_score > best_score:
        best_score = lr_score
        best_lambd = l
print("best score LogisticRegression",best_score)
print("best lambda",best_lambd)

<a name='1.2.2'></a>
### 2.2 - Ridge

- Ridge est implémenté en ajoutant un terme de régularisation L2 à la fonction de coût. Ce terme de régularisation est le produit de la somme des carrés du vecteur de poids $w$ et d'un paramètre d'ajustement $\lambda$. Par rapport à la régularisation L1, la régularisation L2 a tendance à rendre les coefficients de poids proches de zéro mais pas égaux à zéro, elle ne fait donc pas disparaître complètement le coefficient de caractéristique.
$$J = L+s = L + \frac{\lambda}{n}\sum_{i}w_i^2$$

- La caractéristique principale de Ridge est qu'elle peut réduire la corrélation entre les caractéristiques et qu'elle est insensible aux valeurs aberrantes.

<a name='exo-2'></a>
#### Exercice 2 : Régularisation L2
**Complétez cette partie du code en vous basant sur le code ci-dessus.**

In [ ]:
## Régularisation L2
# l in [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]
# dw = 1/n * np.dot(X,(hat_Y-Y).T) + (lambd / n) * 2 * w




**Output attendu** :
<table>
    <tr>
        <td>
            best score LogisticRegression
        </td>
        <td>
            72.0
        </td>
    </tr>
    <tr>
        <td>best lambda
        </td>
        <td>
            0.3
        </td>
    </tr>
</table>

<a name='1.2.3'></a>
### 2.3 - Recherche par grille avec validation croisée

L'ensemble de données est divisé en un ensemble d'entraînement et un ensemble de test. La séparation de l'ensemble de données d'entraînement en ensemble de formation et ensemble de validation se fait par validation croisée. Implémentons-le sans utiliser la bibliothèque sklearn pour comprendre le système :

In [ ]:
best_score=0
for lr in [0.001, 0.005, 0.01, 0.05]:
    for ni in [500, 1000, 2000, 3000]:
        cv_scores = []
        for cv in range(5):
            X = train_set_x
            y = train_set_y
            logistic_regression_model = model(np.hstack((X[:, :(cv*40)], X[:, (cv*40)+40:])), np.hstack((y[:, :(cv*40)], y[:, (cv*40)+40:])), 
                                              X[:,(cv*40):(cv*40+40)], y[:,(cv*40):(cv*40+40)], num_iterations=ni, learning_rate=lr, print_cost=False)
            cv_score = logistic_regression_model["res"]
            cv_scores.append(cv_score)
        mean_score=np.mean(cv_scores)
        print("num_iterations: ", ni, "learning_rate: ", lr,'res: {:.3f}'.format(mean_score))
        if mean_score > best_score:
            best_score = mean_score
            best_score_combination = (ni, lr)
print("best score LogisticRegression",best_score)
print("best num_iterations and learning_rate",best_score_combination)

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=1000, learning_rate=0.01, print_cost=True)

<a name='1.2.2'></a>
### 2.4 - Recherche aléatoire avec validation croisée
<img src="Cross-validation.png" style="width:500px">

<a name='exo-3'></a>
#### Exercice 3 : Recherche aléatoire avec validation croisée
**Complétez cette partie du code en vous basant sur le code ci-dessus.**

In [ ]:
## Recherche aléatoire avec validation croisée
# 16 points verifiés
# 0.001 <= lr < 0.05
# 500 <= ni < 3000
# Validation croisée à 5 blocs (40 échantillons dans chaque bloc)

np.random.seed(1)





**Output attendu** :
<table>
    <tr>
        <td>
            best score LogisticRegression
        </td>
        <td>
            62.0
        </td>
    </tr>
    <tr>
        <td>
            best num_iterations and learning_rate
        </td>
        <td>
            (1460, 0.0010056043660498994)
        </td>
    </tr>
</table>

=========================================================
=========================================================
<a name='2'></a>
# II - Réalisation à l'aide de Scikit Learn

<a name='2.1'></a>
## 1 - Optimisation

<a name='2.1.1'></a>
### 1.1 - Recherche par grille

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin

class ModelWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, num_iterations=2000, learning_rate=0.5):
        self.num_iterations = num_iterations
        self.learning_rate = learning_rate
        
    def fit(self, X, y):
        self.model = model(X.T, y.T, test_set_x, test_set_y, num_iterations=self.num_iterations, learning_rate=self.learning_rate, print_cost=False)
        self.classes_ = np.unique(y)
        
    def predict(self, X):
        Y_prediction = predict(self.model["w"], self.model["b"], X.T)
        return Y_prediction.T
    
    def score(self, X, y):
        Y_prediction = self.predict(X)
        return 100 - np.mean(np.abs(Y_prediction - y)) * 100

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'num_iterations': [500, 1000, 2000, 3000],
    'learning_rate': [0.001, 0.005, 0.01, 0.05]
}

grid_search = GridSearchCV(ModelWrapper(), param_grid, scoring='accuracy', cv=3)
grid_search.fit(train_set_x.T, train_set_y.T.ravel())

print("Grid Search best parameters:", grid_search.best_params_)
print("Grid Search best score:", grid_search.best_score_)

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=1000, learning_rate=0.005, print_cost=True)

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=1000, learning_rate=0.01, print_cost=True)

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=500, learning_rate=0.005, print_cost=True)

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=500, learning_rate=0.001, print_cost=True)

<a name='2.1.2'></a>
### 1.2 - Recherche aléatoire

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'num_iterations': [500, 1000, 2000, 3000, 4000, 5000],
    'learning_rate': [0.001, 0.005, 0.01, 0.05]
}

random_search = RandomizedSearchCV(ModelWrapper(), param_dist, scoring='accuracy', cv=3, n_iter=10, random_state=42)
random_search.fit(train_set_x.T, train_set_y.T.ravel())

print("Random Search best parameters:", random_search.best_params_)
print("Random Search best score:", random_search.best_score_)

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=5000, learning_rate=0.05, print_cost=True)

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=2000, learning_rate=0.05, print_cost=True)

<a name='2.1.3'></a>
### 1.3 - Optimisation bayésienne

In [ ]:
from skopt import BayesSearchCV

opt = BayesSearchCV(
    ModelWrapper(),
    {
        'num_iterations': (500, 5000),
        'learning_rate': (0.001, 0.05)
    },
    n_iter=10,
    scoring='accuracy',
    cv=3
)

opt.fit(train_set_x.T, train_set_y.T.ravel())

print("Bayesian Optimization best parameters:", opt.best_params_)
print("Bayesian Optimization best score:", opt.best_score_)

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=1026, learning_rate=0.022592522618279955, print_cost=True)

<a name='2.2'></a>
## 2 - le surapprentissage

<a name='2.2.1'></a>
### 2.1 - LASSO et Ridge

In [ ]:
from sklearn.linear_model import Ridge

## Définir l'espace des paramètres
param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0],
    'max_iter': [500, 1000, 2000, 3000]
}

## Créer un modèle Ridge
ridge_model = Ridge()

## Intégration de méthodes dans GridSearchCV
grid_search = GridSearchCV(ridge_model, param_grid, scoring='accuracy', cv=5, verbose=1, n_jobs=-1)

## Effectuer une recherche dans la grille
grid_search.fit(train_set_x.T, train_set_y.T)

## Imprimer les meilleurs paramètres et le meilleur score
print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

## Créer une instance de modèle Ridge en utilisant les meilleurs paramètres obtenus
best_alpha = grid_search.best_params_['alpha']
best_max_iter = grid_search.best_params_['max_iter']
ridge_model = Ridge(alpha=best_alpha, max_iter=best_max_iter)

## Modèle de formation
ridge_model.fit(train_set_x.T, train_set_y.T)

## Prédiction
Y_pred = ridge_model.predict(train_set_x.T)
100 - np.mean(np.abs(Y_pred - test_set_y)) * 100

=========================================================
=========================================================
# GAN


<a name='3'></a>
## 3 - Réseaux Adverses Génératifs (GAN)

<img src="GAN.png" style="width:450px">

<a name='exo-4'></a>
**Exercice 4** Implémentez la partie Générateur décrite dans la Figure.

**Instructions**:
1. Structure: Couche d'entrée → Couche cachée (activation $\tanh$) → Couche de sortie (activation $\tanh$)
2. `forward()`: Reçoit un bruit d'entrée $z$ et génère un échantillon faux
3. `backward()`: Calcule les gradients (règle de la chaîne)
    - `dz2`: Gradient de la fonction de perte par rapport à la sortie du générateur
    - `dh2 = dz2 * (1 - h2²)`: Dérivée de tanh
    - Rétropropagation du gradient couche par couche

In [ ]:
class Generator:
    def __init__(self, input_dim, hidden_dim, output_dim):
        ########## 4 lignes #########
        
        #############################

    def forward(self, z):
        self.z = z  # Sauvegarder les données d'entrée
        ########## 4 lignes #########
        
        #############################
        return self.h2

    def backward(self, dz2):
        ########## 6 lignes #########
        
        #############################
        return dW1, db1, dW2, db2

**Questions supplémentaires:**

1. Pourquoi le générateur utilise-t-il `tanh` comme fonction d'activation de la couche de sortie? Quelle est sa plage de sortie?

2. Dans `backward()`, quelle est la base mathématique de la ligne `da1 = np.dot(da2, self.W2.T) * (1 - self.a1**2)`?

3. Si on remplace la fonction d'activation de la couche cachée par ReLU, comment devrait-on modifier le code? Quels seraient les impacts?

<a name='3.2'></a>
## 3.2 - Discriminateur

<a name='exo-5'></a>
**Exercice 5** Implémentez la partie Discriminateur décrite dans la Figure.

**Instructions**:
1. Structure: Couche d'entrée → Couche cachée (tanh) → Couche de sortie (sigmoid)
2. `forward()`: Sort la probabilité que l'échantillon soit réel (entre 0 et 1)
3. `backward()`: Calcule le gradient de l'entropie croisée binaire
    - `dh2 = y_pred - y_true`: Propriété spéciale de la combinaison sigmoid + entropie croisée
    - Propagation du gradient similaire au générateur

In [ ]:
class Discriminator:
    def __init__(self, input_dim, hidden_dim, output_dim):
        ########## 4 lignes #########
        
        #############################

    def forward(self, x):
        self.x = x  # Sauvegarder les données d'entrée
        ########## 4 lignes #########
        
        #############################
        return self.h2

    def backward(self, y_true, y_pred):
        ########## 6 lignes #########
        
        #############################
        return dW1, db1, dW2, db2

**Questions supplémentaires:**

1. Pourquoi le discriminateur utilise-t-il une sigmoïde comme fonction d'activation de la couche de sortie?
2. Expliquez la dérivation mathématique de `dh2 = y_pred - y_true` (indice: perte d'entropie croisée + dérivée de sigmoid)
3. Si on n'utilise pas sigmoid au dernier étage du discriminateur, comment devrait-on ajuster la fonction de perte?

<a name='3.3'></a>
## 3.3 - Boucle d'entraînement du GAN

Ici, on étude un cas pratique et fondamental des GAN : l'apprentissage d'une distribution de données unidimensionnelle simple. Plus spécifiquement, elle démontre comment un GAN peut apprendre à générer des données qui imitent une distribution gaussienne (normale) standard. 

<a name='exo-6'></a>
**Exercice 6** Réaliser l'entraînement du GAN.

**Instructions**:
1. Entraînement du discriminateur:
    - Mélanger les échantillons réels (étiquette 1) et générés (étiquette 0)
    - Minimiser la perte d'entropie croisée: $-\mathbb{E}[\log(D(x)) + \log(1-D(G(z)))]$
2. Entraînement du générateur:
    - Objectif: Maximiser $\log(D(G(z)))$
    - Calcul du gradient: `dz2 = -1/(D(G(z))` (dérivation dans les exercices)
3. Stratégie d'entraînement alterné

In [ ]:
def train_gan(generator, discriminator, num_epochs, batch_size, learning_rate):
    ## Distribution de données réelles N(0,1)
    real_data = np.random.normal(0, 1, (1000, 1))

    for epoch in range(num_epochs):
        for _ in range(len(real_data) // batch_size):
            ## échantillons réels
            ########## 2 lignes #########
            
            #############################

            # échantillons générés
            z = np.random.normal(0, 1, (batch_size, 1))
            ########## 2 lignes #########
            
            #############################

            ## Combiner échantillons réels et générés
            ########## 2 lignes #########
            
            #############################

            ## 1. Entraîner le discriminateur
            ######## 7~10 lignes ########
            
            #############################

            ## 2. Entraîner le générateur
            z = np.random.normal(0, 1, (batch_size, 1))
            ########## 3 lignes #########
            
            #############################

            ## Perte et gradient du générateur
            ######### 6~9 lignes ########
            
            #############################

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, D Loss: {d_loss}, G Loss: {g_loss}")

    # Échantillons générés vs distribution réelle
    z = np.random.normal(0, 1, (1000, 1))
    generated_samples = generator.forward(z)
    plt.hist(generated_samples, bins=30, alpha=0.5, label="Generated")
    plt.hist(real_data, bins=30, alpha=0.5, label="Real")
    plt.legend()
    plt.show()

# Initialisation du réseau
generator = Generator(input_dim=1, hidden_dim=10, output_dim=1)
discriminator = Discriminator(input_dim=1, hidden_dim=10, output_dim=1)

# Entraîner le GAN
train_gan(generator, discriminator, num_epochs=1000, batch_size=32, learning_rate=0.01)

**Questions supplémentaires:**
1. Pourquoi ajoute-t-on `1e-7` dans le calcul de la perte? Quel est le risque si on le supprime?
2. Dérivez mathématiquement le gradient du générateur `dz2 = -1/(d_preds_fake + 1e-7)`
3. Si le discriminateur est "trop bon" ($D(G(z))\rightarrow 0$), quel est l'impact sur le gradient du générateur? Comment résoudre ce problème?
4. Dans le code, on entraîne d'abord le discriminateur puis le générateur. Que se passerait-il si on inversait l'ordre?

Avec ce cas-là:
1. **Illustration du principe fondamental** :
    Même avec une architecture minimaliste, le GAN capture l'essence de l'apprentissage génératif
2. **Diagnostic facile** :
    Les problèmes typiques des GAN (mode collapse, instabilité) sont immédiatement visibles :
    - Si l'histogramme généré ne ressemble pas à une gaussienne
    - Si les pertes divergent au lieu de converger vers 0.693
3. **Base pour des extensions** :
    Comprendre ce cas permet d'aborder des applications complexes (Extension potentielle à MNIST) :
    
    ```generator = Generator(input_dim=100, hidden_dim=256, output_dim=784)  # 28x28 images```
    
    ```discriminator = Discriminator(input_dim=784, hidden_dim=256, output_dim=1)```

**Questions générales:**:

1. Si les échantillons générés et la distribution réelle ne se chevauchent pas du tout, quelles en sont les causes possibles?

2. Comment modifier le code pour implémenter un GAN conditionnel (générer des données d'une catégorie spécifique)?